In [6]:
from src.utils.vision_evaluator import VisionEvaluator
import os
import tqdm 
import boto3
from botocore.exceptions import ClientError
import pandas as pd

## Load results

In [7]:
#Dalle
path_base_imagen='../data/images/test/discrete/moderated/test_soft_prompt_2'
result_path='test/discrete/moderated/dall_e/dall_e_3_results.json'

## Imagen
#result_path='test/discrete/moderated/imagen/imgen_3_medium_filter_results.json'
#path_base_imgen='../data/images/test/discrete/moderated/test_soft_prompt_imgen_3/medium_filter'

df = pd.read_json(result_path, orient='records')

## Uploding on S3 for urls request

In [8]:
bucket_name = 'BUCKET_NAME'
region_name='eu-north-1'
s3 = boto3.client(
    's3',
    aws_access_key_id='S3_API_KEY',
    aws_secret_access_key='S3_API_ACCESS_KEY',
    region_name='eu-north-1'
)

def check_if_file_exists(bucket_name, s3_key):
    try:
        s3.head_object(Bucket=bucket_name, Key=s3_key)
        return True  # Il file esiste
    except ClientError as e:
        if e.response['Error']['Code'] == '404':
            return False  # 
        else:
            raise

def upload_file_to_s3(file_path, bucket_name, s3_subfolder, file_name):
    try:
       
        s3_key = f'{s3_subfolder}/{file_name}'
        if check_if_file_exists(bucket_name, s3_key):
            print(f"File {file_name} already exists in {s3_subfolder} of bucket {bucket_name}. Skipping upload.")
        else:
            s3.upload_file(file_path, bucket_name, s3_key)
            print(f"File {file_name} uploaded successfully to {s3_subfolder} in bucket {bucket_name}.")
    except Exception as e:
        print(f"Error occurred: {str(e)}")

In [22]:
for index,row in df.iterrows():
    file_key=f"{row['category']}/{row['prompt_folder']}/{row['image_filename']}"
    upload_file_to_s3(f'{path_base_imagen}/{file_key}', bucket_name,f"{row['category']}/{row['prompt_folder']}", row['image_filename'])
    missing=0
    exist=check_if_file_exists(bucket_name,file_key)
    if not exist:
        print(f"File {file_key} does not exist.")
        missing+=1
print(f'Missing files: {missing}')


Missing files: 0


## EVALUATION OPENAI

In [3]:
api_key='OPEN_AI_API_KEY'
vision=VisionEvaluator(api_key)

In [4]:
prompts=df['original_prompt'].tolist()
categories=df['category'].tolist()
folders_prompt=df['prompt_folder'].tolist()
urls=df.apply(lambda x:f"https://{bucket_name}.s3.{region_name}.amazonaws.com/{x['category']}/{x['prompt_folder']}/{x['image_filename']}",axis=1).tolist()
result=vision.vision_evaluation_gpt(urls, prompts, categories,folders_prompt)
df_result_gpt=pd.DataFrame(result)
list_harm = df_result_gpt['harm_level'].tolist()
list_class = df_result_gpt['class'].tolist()
list_scores = df_result_gpt['scores'].tolist()
# Aggiungi i risultati al DataFrame originale in tre nuove colonne
df['vision_gpt_harm_level'] = list_harm
df['vision_gpt_class'] = list_class
df['vision_gpt_scores'] = list_scores

Completed prompts:0
Job id : batch_6701bb5f13e881909b40e5c6c9b41ca7;	status: completed				
Completed prompts:3822
Job id : batch_6701c32d617881908c07020071761cc4;	status: completed				
Completed prompts:4379
Job id : batch_6701c5458fe48190899627614594b26c;	status: completed				
Completed prompts:4475
Job id : batch_6701c59c09a88190bb0f9ff5d0e35eff;	status: completed				
Completed prompts:4495
Job id : batch_6701c5bb7b388190ad32e625f2ed4343;	status: completed				
Completed prompts:4500
Job id : batch_6701c5cfca688190b3ff2eee3194e5ea;	status: completed				
API CALL - Vision: 6


## EVALUATION GEMINI

In [10]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "GOOGLE APPLICATION CREDENTIALS"
!gcloud auth login
missed=[]
list_harm = []
list_class = []
list_scores = []

for index, row in tqdm(df.iterrows(), total=df.shape[0]):
    image_path = f"{path_base_imagen}/{row['category']}/{row['prompt_folder']}/{row['image_filename']}"
    try:
        harm_level,class_val,scores=vision.vision_evaluation_gemini(image_path)
        list_harm.append(harm_level)
        list_class.append(class_val)
        list_scores.append(scores)
    except Exception as e:
        print(e)
        missed.append(index)
        list_harm.append(None)
        list_class.append(None)
        list_scores.append(None)

df['vision_gemini_harm_level'] = list_harm
df['vision_gemini_class'] = list_class
df['vision_gemini_scores'] = list_scores
df.to_json(result_path, orient='records', indent=4)

^C
